# Z-Image-Turbo — First Test on Kaggle T4
**Test prompt:** *a stickman sleeping in bed*

### Before running: right panel -> Settings
1. **Accelerator -> GPU T4 x2** (this test uses 1 GPU; the 2nd is for later batching)
2. **Internet -> ON** (needed to download the model, ~10–12 GB first run)

Then run cells top to bottom. **What we're measuring:** seconds/image, peak VRAM, black-image check, and whether a raw 5-word prompt beats an engineered one.

> **T4 note:** this notebook auto-uses the SDNQ-INT8 build + FP32 compute on T4 because BF16 is slow/emulated there and FP16 gives black images in Z-Image. On Ampere+ GPUs it uses the original BF16 model.


In [ ]:
# --- Cell 1: GPU check + install (ZImagePipeline needs recent diffusers) ---
!nvidia-smi
!pip install -q --upgrade "git+https://github.com/huggingface/diffusers.git" sdnq
print("Install done. If the import in Cell 3 fails, restart the session (Run -> Restart) and rerun from Cell 2.")
# diffusers>=0.41 needs torchao>=0.15 (FqnToConfig); Kaggle bundles 0.14.x.
# --no-deps so pip cannot disturb Kaggle's torch. If imports still fail,
# Cell 3's shim covers torchao 0.14, or run: !pip uninstall -y torchao
!pip install -q --upgrade --no-deps "torchao>=0.15.0"


In [ ]:
# --- Cell 2: CONFIG — change these and rerun Cells 4-5 anytime ---
NUM_IMAGES = 4          # 1 raw-prompt + 3 engineered-prompt variations
BASE_SEED  = 42
SIZE       = 1024       # drop to 768 if you ever hit CUDA OOM
STEPS      = 9          # 9 steps = 8 DiT forwards (official Turbo setting)
MODEL_MODE = "auto"     # auto | int8 | uint4 | original
GPU_ID     = 0

RAW_PROMPT = "a stickman sleeping in bed"
ENGINEERED_PROMPT = (
    "Minimalist black stick-figure line drawing on a plain white background, "
    "children's doodle style, clean thin black outlines only: one simple "
    "stickman sleeping peacefully in a bed, lying on his back, round head "
    "resting on a rectangular pillow, body hidden under a simple rectangular "
    "blanket, eyes drawn as two small closed curves, three handwritten letter "
    "'Z' floating above his head growing larger, a simple bedside table with "
    "a small lamp beside the bed, a crescent moon in a square window, "
    "lots of empty white space, flat 2D doodle, black ink on white paper"
)
print("Config set.")


In [ ]:
# --- Cell 3: imports + load the pipeline (first run downloads weights) ---
import json, time
from pathlib import Path
import torch

assert torch.cuda.is_available(), "No GPU! Settings -> Accelerator -> GPU T4 x2"
print("GPU:", torch.cuda.get_device_name(GPU_ID),
      "| compute:", torch.cuda.get_device_capability(GPU_ID),
      "| visible GPUs:", torch.cuda.device_count(),
      "| torch:", torch.__version__)

REPOS = {"original": "Tongyi-MAI/Z-Image-Turbo",
         "int8": "Disty0/Z-Image-Turbo-SDNQ-int8",
         "uint4": "Disty0/Z-Image-Turbo-SDNQ-uint4-svd-r32"}

mode = MODEL_MODE
if mode == "auto":
    major, _ = torch.cuda.get_device_capability(GPU_ID)
    mode = "original" if major >= 8 else "int8"   # T4 = sm_75 -> SDNQ INT8 path
print("Model mode:", mode, "->", REPOS[mode])

# Compat shim: torchao<0.15 (Kaggle) lacks FqnToConfig, which
# diffusers>=0.41 imports unconditionally. Alias the old name (we use
# SDNQ, never torchao quantization, so this only unblocks the import).
try:
    import torchao.quantization as _tq
    if not hasattr(_tq, "FqnToConfig") and hasattr(_tq, "ModuleFqnToConfig"):
        _tq.FqnToConfig = _tq.ModuleFqnToConfig
        print("[compat] aliased ModuleFqnToConfig -> FqnToConfig")
except Exception:
    pass

from diffusers import ZImagePipeline
t0 = time.time()
if mode == "original":
    pipe = ZImagePipeline.from_pretrained(REPOS[mode], torch_dtype=torch.bfloat16,
                                          low_cpu_mem_usage=False)
    pipe.to(f"cuda:{GPU_ID}")
else:
    from sdnq import SDNQConfig  # noqa: F401  (registers SDNQ; must precede load)
    from sdnq.common import use_torch_compile as triton_is_available
    from sdnq.loader import apply_sdnq_options_to_model
    # FP32 compute on purpose: FP16 -> black images; BF16 breaks/slows Triton on T4.
    pipe = ZImagePipeline.from_pretrained(REPOS[mode], torch_dtype=torch.float32)
    if triton_is_available:
        pipe.transformer  = apply_sdnq_options_to_model(pipe.transformer,  use_quantized_matmul=True)
        pipe.text_encoder = apply_sdnq_options_to_model(pipe.text_encoder, use_quantized_matmul=True)
        print("Triton INT8 quantized matmul: ON")
    else:
        print("WARNING: Triton unavailable -> slow dequant path")
    try:
        pipe.enable_model_cpu_offload(gpu_id=GPU_ID)
    except TypeError:
        pipe.enable_model_cpu_offload()

for fn in ("enable_vae_slicing", "enable_vae_tiling"):
    try: getattr(pipe, fn)()
    except Exception: pass

print(f"Pipeline loaded in {time.time()-t0:.0f}s")


In [ ]:
# --- Cell 4: helpers ---
from PIL import Image, ImageDraw
from IPython.display import display
import numpy as np

OUT = Path("/kaggle/working/z_image_turbo_test_outputs")
OUT.mkdir(parents=True, exist_ok=True)

def generate_one(prompt, seed):
    """Generate one image; returns (PIL image, seconds, peak_vram_gb)."""
    torch.cuda.reset_peak_memory_stats(GPU_ID)
    torch.cuda.synchronize(GPU_ID); t = time.time()
    img = pipe(prompt=prompt, height=SIZE, width=SIZE,
               num_inference_steps=STEPS, guidance_scale=0.0,  # Turbo: always 0.0
               generator=torch.Generator("cuda").manual_seed(seed)).images[0]
    torch.cuda.synchronize(GPU_ID)
    return img, time.time()-t, torch.cuda.max_memory_allocated(GPU_ID)/1024**3

def check_black(img):
    a = np.asarray(img.convert("RGB"), dtype=np.float32)
    return float(a.mean()), float(a.std())

def contact_sheet(paths, labels, thumb=512):
    cols = 2 if len(paths) > 1 else 1
    rows = (len(paths)+cols-1)//cols; footer = 28
    sheet = Image.new("RGB", (cols*thumb, rows*(thumb+footer)), "white")
    d = ImageDraw.Draw(sheet)
    for i,(p,lab) in enumerate(zip(paths,labels)):
        x,y = (i%cols)*thumb, (i//cols)*(thumb+footer)
        sheet.paste(Image.open(p).convert("RGB").resize((thumb,thumb)), (x,y))
        d.text((x+8,y+thumb+6), lab, fill="black")
    return sheet

print("Helpers ready. Output dir:", OUT)


In [ ]:
# --- Cell 5: RUN THE TEST (rerun this cell after changing Cell 2 config) ---
jobs = [("raw", RAW_PROMPT, BASE_SEED)]
while len(jobs) < NUM_IMAGES:
    jobs.append(("engineered", ENGINEERED_PROMPT, BASE_SEED + len(jobs)))

results, paths, labels = [], [], []
for i,(label,prompt,seed) in enumerate(jobs):
    img, secs, peak = generate_one(prompt, seed)
    p = OUT / f"stickman_{i:02d}_{label}_seed{seed}.png"
    img.save(p)
    mean, std = check_black(img)
    black = mean < 2.0 and std < 2.0
    print(f"[{i+1}/{len(jobs)}] {p.name} | {secs:.1f}s | peak {peak:.2f} GB | "
          f"mean {mean:.1f} std {std:.1f}" + ("  <-- BLACK IMAGE!" if black else ""))
    results.append({"label":label,"prompt":prompt,"seed":seed,"file":str(p),
                    "seconds":round(secs,2),"peak_vram_gb":round(peak,2),
                    "pixel_mean":round(mean,2),"pixel_std":round(std,2),
                    "black_image_suspected":black})
    paths.append(p); labels.append(f"{label} | seed {seed} | {secs:.0f}s")

sheet = contact_sheet(paths, labels)
sheet.save(OUT/"contact_sheet.png")
display(sheet)

times = [r["seconds"] for r in results]
steady = sum(times[1:])/len(times[1:]) if len(times) > 1 else times[0]
summary = {"model_mode":mode, "model_repo":REPOS[mode], "size":SIZE, "steps":STEPS,
           "per_image_seconds":times, "steady_state_seconds":round(steady,2),
           "images_per_hour_per_gpu":round(3600/steady),
           "max_peak_vram_gb":max(r["peak_vram_gb"] for r in results), "images":results}
(OUT/"run_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps({k:v for k,v in summary.items() if k != "images"}, indent=2))
print("Saved to", OUT, "-> see the Output tab. Send me contact_sheet.png + these timings!")


## Playground
Model stays loaded — try your own prompts without reloading (uncomment / edit):


In [ ]:
# --- Cell 6: playground ---
# img, secs, peak = generate_one("a stickman riding a bicycle, simple black line doodle on white paper", seed=123)
# display(img); print(f"{secs:.1f}s | peak {peak:.2f} GB")


## Troubleshooting
- **`ImportError: ZImagePipeline`** -> rerun Cell 1, then **restart the session**, rerun from Cell 2.
- **CUDA out of memory** -> set `SIZE = 768` in Cell 2, restart session, rerun all.
- **Black images** -> do NOT switch dtype to float16 (that's the cause). Keep the auto SDNQ/FP32 path and share `run_summary.json`.
- **Very slow first image** -> normal: weight download + first-call warmup. Judge speed from image #2 onwards (`steady_state_seconds`).
- **Session died / outputs gone** -> use *Save Version -> Save & Run All* so outputs in `/kaggle/working` persist as notebook output.

## Next steps (after this test)
1. Note `steady_state_seconds`, `max_peak_vram_gb`, and how the stickman looks (extra limbs? bed correct? doodle style held?).
2. With the real s/image we can compute the batch plan for hundreds across **both** T4s.
3. Then add the Qwen3-VL quality-check stage (prompt match + extra-limb rejection).
- **`cannot import name 'FqnToConfig' from 'torchao.quantization'`** -> torchao<0.15 vs diffusers>=0.41 clash. Run `!pip install -q --upgrade --no-deps "torchao>=0.15.0"` (Cell 1 already does this), or `!pip uninstall -y torchao`, then rerun Cell 3. The shim in Cell 3 also covers torchao 0.14 directly.
